In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

In [3]:
df = pd.read_csv("global_pharmacy_sales_2020_2025_daily_dataset.csv")

print(df.head())
print(df.shape)

         date  year  month  day         region    country category  \
0  2025-10-12  2025     10   12    Middle East        UAE  Chronic   
1  2020-09-15  2020      9   15     South Asia      India  Chronic   
2  2020-02-26  2020      2   26    Middle East        UAE  Vitamin   
3  2025-11-09  2025     11    9     South Asia  Sri Lanka  Chronic   
4  2022-04-04  2022      4    4  South America  Argentina  Chronic   

     medicine age_group  units_sold  unit_price  stock_level  \
0  Amlodipine      0-12         328       45.38         4774   
1  Amlodipine     26-45         371       75.49         4584   
2   Vitamin C      0-12         948       22.51         3934   
3  Amlodipine      0-12         275       63.29         4544   
4  Amlodipine     26-45         563       44.28         4284   

   expiry_days_remaining  covid_flag  
0                    466           0  
1                    181           1  
2                    556           1  
3                    330           0  

In [4]:
print(df.columns)

Index(['date', 'year', 'month', 'day', 'region', 'country', 'category',
       'medicine', 'age_group', 'units_sold', 'unit_price', 'stock_level',
       'expiry_days_remaining', 'covid_flag'],
      dtype='object')


In [5]:
# create sales target
df['sales'] = df['units_sold'] * df['unit_price']
print(df[['units_sold', 'unit_price', 'sales']].head())

   units_sold  unit_price     sales
0         328       45.38  14884.64
1         371       75.49  28006.79
2         948       22.51  21339.48
3         275       63.29  17404.75
4         563       44.28  24929.64


In [6]:
#now we create features without using sales
# stock and price
df['stock_value'] = df['stock_level'] * df['unit_price']

# stock and expiry
df['expiry_stock'] = df['stock_level'] * df['expiry_days_remaining']

# price during covid
df['covid_price'] = df['unit_price'] * df['covid_flag']

# stock during covid
df['covid_stock'] = df['stock_level'] * df['covid_flag']

# expiry during covid
df['covid_expiry'] = df['expiry_days_remaining'] * df['covid_flag']

In [7]:
print(df[['stock_value',
          'expiry_stock',
          'covid_price',
          'covid_stock',
          'covid_expiry']].head())

   stock_value  expiry_stock  covid_price  covid_stock  covid_expiry
0    216644.12       2224684         0.00            0             0
1    346046.16        829704        75.49         4584           181
2     88554.34       2187304        22.51         3934           556
3    287589.76       1499520         0.00            0             0
4    189695.52       2527560         0.00            0             0


In [8]:
# input features
x = df[['stock_level',
        'expiry_days_remaining',
        'covid_flag',
        'stock_value',
        'expiry_stock',
        'covid_price',
        'covid_stock',
        'covid_expiry']]

# target
y = df['sales']

In [9]:
# trfaining and testing data split 
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

print("training data:", x_train.shape)
print("testing data:", x_test.shape)

training data: (142392, 8)
testing data: (35598, 8)


In [10]:
#create Random Forest model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

In [11]:
# perform 5 fold cross validation
scores = cross_val_score(
    model,
    x_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("cv scores:", scores)
print("average cv score:", scores.mean())
#5-fold cross-validation divides the training data into 5 parts, model is tested multiple times and the average R² score shows its performance.

cv scores: [0.9291616  0.93480561 0.92400867 0.93185363 0.93641214]
average cv score: 0.9312483309928339


In [12]:
#train the model 
model.fit(x_train, y_train)
print("model trained")

model trained


In [13]:
# feature inportance it tells which feature is contribute the most to predict  salse 
imp = pd.DataFrame({
    'feature': x.columns,
    'importance': model.feature_importances_
})

imp = imp.sort_values(
    by='importance',
    ascending=False
)

print(imp)

                 feature  importance
0            stock_level    0.584166
3            stock_value    0.255515
5            covid_price    0.108103
6            covid_stock    0.025249
4           expiry_stock    0.011362
1  expiry_days_remaining    0.010146
7           covid_expiry    0.005143
2             covid_flag    0.000315


In [14]:
# select top features
top = imp.head(5)

print("top features:")
print(top)

top features:
        feature  importance
0   stock_level    0.584166
3   stock_value    0.255515
5   covid_price    0.108103
6   covid_stock    0.025249
4  expiry_stock    0.011362


In [ ]:
# now we train model on selected features
# get selected feature names
cols = top['feature'].tolist()

x2 = df[cols]

# split data
x_train, x_test, y_train, y_test = train_test_split(
    x2,
    y,
    test_size=0.2,
    random_state=42
)

# train model
model.fit(x_train, y_train)


model trained with selected features


In [16]:
scores = cross_val_score(
    model,
    x_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("new cv scores:", scores)
print("new average score:", scores.mean())

new cv scores: [0.93133238 0.93501669 0.92504542 0.93197815 0.93504483]
new average score: 0.9316834943976297
